#  Using assertions


The goal of [defensive programming](https://en.wikipedia.org/wiki/Defensive_programming) is to try to maximize the reliability and overall quality of a piece of software. For us, this means that we should take steps to handle unexpected input values in our code, and to provide helpful error messages that give meaningful guidance to the user when a program raises an exception. We can take a step toward writing more reliable software by using a helpful feature in Python: assertions.

## Assertions

*Assertions* are a way to assert, or ensure, that the values being used in your scripts are going to be suitable for what the code does. Let's start by considering a function `convert_celsius_fahrenheit` that converts temperatures from degrees Celsius to degrees Fahrenheit. We can define and use our function in the cell below.

In [ ]:
# A first version of the function, with no safety checks at all
def convert_celsius_fahrenheit(temperature):
    """Converts temperature in Celsius to Fahrenheit"""
    return (temperature * 9/5) + 32

# Try it out with a sensible temperature
temperature_celsius = 20
temperature_fahrenheit = convert_celsius_fahrenheit(temperature_celsius)

print(f"A temperature of {temperature_celsius}°C is equivalent to {temperature_fahrenheit}°F.")



In a similar vein, consider incorporating a safety check for temperature conversions to ensure the input is within a plausible range for earthly conditions. This check helps prevent absurd values from being processed, reflecting the principle of validating input data for realism and safety.

In [ ]:
def convert_celsius_fahrenheit(temperature):
    """Converts temperature in Celsius to Fahrenheit"""
    # assert stops the function immediately if the condition is False.
    # Here we check the temperature is within the recorded range on Earth (-89.2 to 56.7 degrees Celsius)
    assert -89.2 <= temperature <= 56.7, "Temperature is outside the plausible Earth range."
    return (temperature * 9/5) + 32

# 20 degrees passes the check, so the function runs as normal
temperature_celsius = 20
temperature_fahrenheit = convert_celsius_fahrenheit(temperature_celsius)

print(f"A temperature of {temperature_celsius}°C is equivalent to {temperature_fahrenheit}°F.")


To mirror the exploration of handling unexpected or invalid input in the context of temperature conversion, let's examine the scenario where an implausible temperature for Earth is provided. This approach will demonstrate the assertion's role in safeguarding against unrealistic values.

In [ ]:
temperature_celsius = -100  # An implausibly low temperature for Earth's surface

# try/except lets us catch the AssertionError, so the notebook keeps running
try:
    temperature_fahrenheit = convert_celsius_fahrenheit(temperature_celsius)
    print(f"A temperature of {temperature_celsius}°C is equivalent to {temperature_fahrenheit}°F.")
except AssertionError as e:
    print(f"Error: {e}")


We already added a short message to the assertion. We can make it even more helpful by stating exactly which values are allowed, so that the user understands the constraint.

Here's how you can apply this improvement to the temperature conversion function:

In [ ]:
def convert_celsius_fahrenheit(temperature):
    """Converts temperature in Celsius to Fahrenheit"""
    # The text after the comma is the message shown when the assertion fails,
    # so make it clearly explain what was expected.
    assert -89.2 <= temperature <= 56.7, "Error: Temperature must be between -89.2°C and 56.7°C, the recorded range of air temperature on Earth."
    return (temperature * 9/5) + 32

temperature_celsius = -100  # An implausibly low temperature for Earth's surface
try:
    temperature_fahrenheit = convert_celsius_fahrenheit(temperature_celsius)
    print(f"A temperature of {temperature_celsius}°C is equivalent to {temperature_fahrenheit}°F.")
except AssertionError as e:
    print(f"{e}")


> ⚠️ **Common Pitfall: never put brackets around an assertion**
>
> It is very tempting to write an assertion as though it were a function call:
>
> ```python
> assert(-89.2 <= temperature <= 56.7, "Temperature is out of range")   # WRONG - always passes!
> ```
>
> This does **not** do what you expect. `assert` is a *statement*, not a function, so the brackets bundle the condition and the message into a **tuple** - and a non-empty tuple is always `True`. The check then silently passes no matter what the temperature is, which is worse than having no check at all.
>
> Write it without brackets:
>
> ```python
> assert -89.2 <= temperature <= 56.7, "Temperature is out of range"   # correct
> ```

Assertions serve as a powerful tool for developers to ensure their code behaves as expected during runtime. By explicitly checking conditions that must be true for the code to function correctly, assertions help identify bugs early in the development process. The addition of a custom error message provides clarity on the nature of the issue, making it easier for users or other developers to understand what went wrong and how to correct it. This practice significantly enhances code reliability and maintainability by preventing the propagation of erroneous values and states through the system.

### Multiple assertions

#### A bad example

Enhancing the robustness of our function with additional assertions ensures not only that we're working with the right type of data but also that it adheres to logical constraints. Applying a similar approach to another context, let's refine a function that converts temperatures from Celsius to Fahrenheit by ensuring the input is numeric and falls within a realistic range. This approach guarantees the function's integrity by preemptively validating input data.

In [ ]:
def convert_celsius_fahrenheit(temperature):
    """Converts temperature in Celsius to Fahrenheit"""
    # First assertion: check that the input is a number at all
    assert isinstance(temperature, (int, float)), "Temperature values must be numbers"
    # Second assertion: check that the value is physically plausible
    assert -89.2 <= temperature <= 56.7, "Temperature must be between -89.2°C and 56.7°C, the recorded range of air temperature on Earth."
    return (temperature * 9/5) + 32

try:
    temperature_celsius = "freezing"  # An invalid string input
    temperature_fahrenheit = convert_celsius_fahrenheit(temperature_celsius)
    print(f"A temperature of {temperature_celsius}°C is equivalent to {temperature_fahrenheit}°F.")
except AssertionError as e:
    print(f"{e}")


In this modified example, the first assertion checks whether the input is either an integer or a float, ensuring that the function operates on a valid numerical value. The second assertion verifies that the temperature falls within a reasonable range for Earth. It seems to work well, so what is the problem?

The problem is that we are using an assertion to do a job that Python already does better by itself. If someone passes text into the function, Python would *naturally* raise a `TypeError` - a standard, well-understood error that every Python programmer recognises. By checking the type ourselves, we replace that clear, standard error with an `AssertionError`, which actually says less about what went wrong.

A good rule of thumb: let Python raise its own errors for **wrong data types**, and save assertions for checking that values make **logical sense**.

#### A better example

In [ ]:
def convert_celsius_fahrenheit(temperature):
    """
    Converts temperature in Celsius to Fahrenheit.
    
    Allows TypeError for incompatible data types.
    Uses assertions to ensure input values are within Earth's plausible temperature range.
    """
    # Let Python raise its own TypeError for incompatible data types,
    # then pass it on with a clearer message.
    try:
        fahrenheit = (temperature * 9/5) + 32
    except TypeError as e:
        raise TypeError("Temperature must be a number (int or float).") from e

    # Use an assertion only for the logical range check
    assert -89.2 <= temperature <= 56.7, "Temperature must be within -89.2°C to 56.7°C, Earth's recorded temperature range."

    return fahrenheit

# Example usage. We catch each error type separately so the message can be specific.
try:
    temperature_celsius = 20  # Valid input
    temperature_fahrenheit = convert_celsius_fahrenheit(temperature_celsius)
    print(f"{temperature_celsius}°C is {temperature_fahrenheit}°F.")
except AssertionError as error:
    print(f"Assertion Error: {error}")
except TypeError as error:
    print(f"Type Error: {error}")


This function will convert temperatures from Celsius to Fahrenheit, allowing `TypeError` to occur naturally for incompatible data types, while using assertions to validate that the input values are within a reasonable and scientifically valid range for Earth's climate.

- **Key Points:**
  - **Handling Incompatible Data Types:** The function is designed to let Python's built-in `TypeError` communicate issues with data type compatibility, providing clear, standard feedback when non-numeric input is provided.
  - **Logical Value Checks:** Assertions are strategically used to ensure that the temperature values entered make sense for Earth's environment, helping guide users towards proper function use by setting realistic bounds.

- **Considerations:**
  - The use of assertions for checking input values against realistic bounds is shown here for educational purposes. However, it's important to remember that assertions may not be suitable for all forms of user input validation, especially in production code. Assertions can be bypassed if Python is run with optimizations (using the `-O` flag), which would ignore these checks entirely.

For robust input validation in user-facing applications, consider employing explicit error handling or custom validation logic that remains active regardless of how the Python interpreter is executed.

This approach not only demonstrates good practices in error handling and input validation, but also aligns with the philosophy of using the right tool for the right job, ensuring code reliability and usability.

### Checking user input with `raise`

For values that come from users or data files, the recommended approach is an explicit `if` check that *raises* a standard error. Unlike an assertion, this check can never be switched off:

In [ ]:
def convert_celsius_fahrenheit(temperature):
    """Converts temperature in Celsius to Fahrenheit."""
    # An explicit check that is always active, even when Python runs with the -O flag
    if not -89.2 <= temperature <= 56.7:
        raise ValueError("Temperature must be between -89.2°C and 56.7°C.")
    return (temperature * 9/5) + 32


try:
    convert_celsius_fahrenheit(-100)
except ValueError as error:
    print(f"Value Error: {error}")

## Summary

This lesson introduced **assertions**, a simple tool for *defensive programming* - writing code that checks its own assumptions. You can now:

- Write an assertion with `assert condition, "message"`. If the condition is `False`, Python stops immediately and raises an `AssertionError` showing your message.
- Write **useful failure messages** that explain what was expected, instead of leaving the reader guessing.
- Catch an `AssertionError` with `try` / `except` so your program can respond gracefully instead of stopping.
- Choose the **right tool for the job**: let Python raise its own `TypeError` for the wrong data *type*, and use assertions to check that values make *logical* sense.

| Approach | Best used for |
| --- | --- |
| `assert` | Internal sanity checks while developing - "this value should never be possible". |
| Letting Python raise `TypeError` | Wrong data types, where Python's own message is already clear. |
| `try` / `except` | Reacting to an error gracefully instead of letting the program stop. |

> **Remember:** assertions are a development aid, not a safety net. They can be switched off entirely when Python is run with the `-O` flag, so never rely on them as the only check on important user input.

---
*Sources: this notebook has been inspired by material from the University of Helsinki [Geo-Python course](https://geo-python-site.readthedocs.io/en/latest/course-info/licensing.html), licensed under a [Creative Commons Attribution-ShareAlike 4.0 International licence](https://creativecommons.org/licenses/by-sa/4.0/deed.en). This adapted version is shared under the same licence.*